# 01 — Corpus Collection

Everything the pipeline learns comes from the ARO source tree. This notebook
walks the repository and assembles a single manifest that captures:

- **`.aro` files** from `Examples/` and `ARO-Application/` — real, working programs
- **Proposals** — the authoritative language specification (59 documents)
- **Book chapters** — the full developer guide across six books
- **Swift action files** — source of truth for every action's verbs and prepositions
- **Live CLI output** — `aro syntax`, `aro actions`, `aro examples` from the actual binary

Collecting live CLI output matters: it reflects the *currently installed* runtime,
so generated training data is always in sync with what `aro` actually understands.

**Output:** `../data/01_corpus/manifest.json`

In [9]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import os
import json
import subprocess
from pathlib import Path

ARO_APP_DIR = (SCRIPT_DIR / '../../../ARO-Application/').resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'ARO root:        {ARO_ROOT}')
print(f'ARO-Application: {ARO_APP_DIR}  (exists: {ARO_APP_DIR.exists()})')
print(f'Output:          {DATA_DIR}')

Fine-tuned model not found on HuggingFace, using base: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Config loaded | model=mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
ARO root:  /Users/kris/Projects/ARO/ARO-Train
Pairs:     /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl
Adapters:  /Users/kris/Projects/ARO/ARO-Train/Train/data/adapters
ARO root:        /Users/kris/Projects/ARO/ARO-Train
ARO-Application: /Users/kris/Projects/ARO/ARO-Application  (exists: True)
Output:          /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge


## Check aro CLI

In [10]:
def run_aro(args, timeout=30):
    try:
        r = subprocess.run(['aro'] + args, capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip(), r.stderr.strip(), r.returncode
    except FileNotFoundError:
        return '', 'aro binary not found in PATH', -1
    except subprocess.TimeoutExpired:
        return '', 'timeout', -1

out, err, code = run_aro(['--version'])
print(f'aro: {out or err}')

aro: 0.8.0


## Collect .aro files

In [11]:
aro_files = []
search_roots = [ARO_ROOT / 'Examples']
if ARO_APP_DIR.exists():
    search_roots.append(ARO_APP_DIR)

for root in search_roots:
    found = list(root.rglob('*.aro'))
    print(f'  {root.name}: {len(found)} .aro files')
    aro_files.extend(found)

print(f'Total: {len(aro_files)} .aro files')

  Examples: 124 .aro files
  ARO-Application: 33 .aro files
Total: 157 .aro files


## Collect Proposals, Book, Swift action files

In [12]:
proposals     = sorted((ARO_ROOT / 'Proposals').glob('*.md'))
book_chapters = sorted((ARO_ROOT / 'Book').rglob('*.md')) if (ARO_ROOT / 'Book').exists() else []
swift_actions = sorted((ARO_ROOT / 'Sources' / 'ARORuntime' / 'Actions').rglob('*.swift'))

print(f'Proposals:     {len(proposals)}')
print(f'Book chapters: {len(book_chapters)}')
print(f'Swift actions: {len(swift_actions)}')

Proposals:     59
Book chapters: 122
Swift actions: 33


## Fetch live data from aro CLI

In [13]:
print('Calling aro CLI...')
syntax_out,   _, _ = run_aro(['syntax'])
actions_out,  _, _ = run_aro(['actions'])
examples_out, _, _ = run_aro(['examples'])

print(f'aro syntax:   {len(syntax_out):,} chars')
print(f'aro actions:  {len(actions_out):,} chars')
print(f'aro examples: {len(examples_out):,} chars')

aro syntax:   84 chars
aro actions:  85 chars
aro examples: 86 chars


## Read static reference docs

In [14]:
extra_docs = {}
for doc_path in [ARO_ROOT / 'README.md', ARO_ROOT / 'CLAUDE.md', ARO_ROOT / 'OVERVIEW.md']:
    if doc_path.exists():
        extra_docs[doc_path.name] = doc_path.read_text()
        print(f'  {doc_path.name}: {len(extra_docs[doc_path.name]):,} chars')

  README.md: 12,448 chars
  CLAUDE.md: 23,628 chars
  OVERVIEW.md: 13,053 chars


## Save manifest

In [15]:
manifest = {
    'aro_files':     [str(f) for f in aro_files],
    'proposals':     [str(f) for f in proposals],
    'book_chapters': [str(f) for f in book_chapters],
    'swift_actions': [str(f) for f in swift_actions],
    'extra_docs':    {k: v[:60_000] for k, v in extra_docs.items()},
    'aro_syntax':    syntax_out,
    'aro_actions':   actions_out,
    'aro_examples':  examples_out,
}

manifest_path = DATA_DIR / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

total = len(aro_files) + len(proposals) + len(book_chapters) + len(swift_actions)
print(f'Manifest saved: {manifest_path}')
print(f'Total files indexed: {total}')

Manifest saved: /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/manifest.json
Total files indexed: 371


In [16]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '01_corpus_collection.png'

_cats   = ['ARO files', 'Proposals', 'Book chapters', 'Swift actions', 'Reference docs']
_vals   = [len(aro_files), len(proposals), len(book_chapters), len(swift_actions), len(extra_docs)]
_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(_cats, _vals, color=_colors, edgecolor='white', width=0.6)
for bar, v in zip(bars, _vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(v), ha='center', va='bottom', fontsize=12, fontweight='bold', color='#2c3e50')
ax.set_ylabel('Items collected')
ax.set_title(f'Corpus Collection — {sum(_vals):,} items across {len(_cats)} categories',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(_vals) * 1.18)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

Saved: run/2026-03-29/01_corpus_collection.png
